### np.linalg.norm

In [1]:
import math

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for elem in a:
        result.extend(flatten(elem))
    return result


def _p_norm_1d(lst, ord=2):
    if ord == 1:
        total = 0
        for x in lst:
            total += abs(x)
        return total
    if ord == 2 or ord is None:
        total = 0
        for x in lst:
            total += x * x
        return math.sqrt(total)
    if ord == float('inf'):
        return max(abs(x) for x in lst) if lst else 0.0

    # generalized p-norm: sum |x|**ord ** (1/ord)
    total = 0
    for x in lst:
        total += abs(x) ** ord
    return total ** (1 / ord)


def np_linalg_norm(x, ord=2):
    """
    Pure-Python equivalent of np.linalg.norm for 1-D vector case.
    Input `x` can be scalar, list, or nested; it is flattened to 1-D
    and then the norm is computed.
    """
    shape = get_shape(x)
    if len(shape) == 0:
        return abs(x)

    flat = flatten(x)
    if len(flat) == 0:
        return 0.0

    if ord in (1, 2, None, float('-inf'), float('inf')):
        # for -inf and -1 we map to 1 (since they are matrix norms;
        #  we keep it simple and treat them as vector norms here)
        if ord in (-1, float('-inf')):
            raise ValueError("ord -1/-inf not supported in this vector-only version")
        if ord == float('-inf'):
            raise ValueError("ord -inf not supported in this vector-only version")
        if ord in (float('inf'), float('-inf')):
            ord = float('inf')

    return _p_norm_1d(flat, ord)

### test cases 

In [2]:
print(np_linalg_norm(5))
print(np_linalg_norm([3, 4]))
print(np_linalg_norm([3, -4]))
print(np_linalg_norm([1, 1, 1, 1]))
print(np_linalg_norm([[1, 2], [3, 4]]))
print(np_linalg_norm([[[1, 2]], [[3, 4]]]))
print(np_linalg_norm([[[1, 2]], [[3, 4]]], ord=1))
print(np_linalg_norm([[[1, 2]], [[3, 4]]], ord=float('inf')))
print(np_linalg_norm([1, 2, 3], ord=1))
print(np_linalg_norm([1, 2, 3], ord=2))
print(np_linalg_norm([1, 2, 3], ord=float('inf')))
print(np_linalg_norm([]))

5
5.0
5.0
2.0
5.477225575051661
5.477225575051661
10
4
6
3.7416573867739413
3
0.0


In [3]:
print(np_linalg_norm([[1, 2], [3]]))

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [4]:
print(np_linalg_norm([1, "a"]))

TypeError: can't multiply sequence by non-int of type 'str'

In [5]:
print(np_linalg_norm([1, 2], ord=float('-inf'))) 

ValueError: ord -1/-inf not supported in this vector-only version